# KAMA - ajuster la reconnaissance vocale sur le pidgin camerounais (Common Voice)
Avant de lancer : Execution > Modifier le type d'execution > GPU (T4 suffit). Le zip `KAMA_pidgin_stt_drive.zip` doit etre dans ton Drive, dossier `KAMA/Dataset/`.
Les donnees Common Voice restent sur ton Drive prive : ne les partage pas et ne les mets pas dans le depot (conditions de Common Voice).
Ordre : 1 -> 5. Pour changer de modele, relance seulement les cellules 4 et 5 avec un autre `MODELE`.

In [ ]:
# 1. Drive et GPU
from google.colab import drive
drive.mount('/content/drive')
import torch
assert torch.cuda.is_available(), 'Pas de GPU : Execution > Modifier le type d\'execution'
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Decompression (toujours refaite, pour ne jamais reutiliser d'anciennes donnees)
import os, shutil, zipfile, json
ZIP = '/content/drive/MyDrive/KAMA/Dataset/KAMA_pidgin_stt_drive.zip'
assert os.path.exists(ZIP), 'zip introuvable : ' + ZIP
os.chdir('/content')
shutil.rmtree('/content/KAMA', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content')
D = '/content/KAMA/stt'
for nom in ['cv_train', 'cv_dev', 'cv_test', 'soreva_test']:
    n = sum(1 for _ in open(f'{D}/manifeste_{nom}.jsonl', encoding='utf-8'))
    print(nom, n)
    assert n > 0
assert os.path.exists(f'{D}/entrainer_stt.py')

In [ ]:
# 3. Bibliotheques
!pip -q install jiwer soundfile accelerate 'transformers>=4.45,<5'  # v5 supprime warmup_ratio : le script garde l'API v4

In [ ]:
# 4. Entrainement. MODELE : 'wbert' (conseille, essaye en premier), 'xlsr' (plus leger) ou 'mms' (adaptateur)
MODELE = 'wbert'
SORTIE = f'/content/sortie_{MODELE}'
!python /content/KAMA/stt/entrainer_stt.py --donnees /content/KAMA/stt --sortie {SORTIE} --modele {MODELE}

In [ ]:
# 5. Resultats, puis copie du seul modele final sur le Drive (pas des points de reprise : le Drive est petit)
import json, os, shutil
r = json.load(open(f'{SORTIE}/resultats.json'))
print('AVANT ', r.get('avant'))
print('APRES ', r['apres'])
dest = f'/content/drive/MyDrive/KAMA/stt_{MODELE}'
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree(f'{SORTIE}/final', dest + '/final')
shutil.copy(f'{SORTIE}/resultats.json', dest)
print('copie sur le Drive :', dest)